# E4: Physics-Informed Neural Network for DNA Thermodynamics

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E4  
**Thesis Chapter:** Chapter 5 — Physical Inductive Bias  

## Core Idea

Standard models (E0–E3) treat ΔH and Tm as two **independent** regression targets, ignoring the physical law that connects them. In reality, for an intramolecular hairpin:

$$T_m = \frac{\Delta H}{\Delta S} - 273.15 \quad \text{(°C)}$$

where ΔS (entropy) is **never directly measured** in the NNN dataset — only ΔH and T_m are labelled.

We force the network to respect this law by:
1. Changing the output head to predict **[ΔH, ΔS]** (two physical quantities)
2. Computing **T_m** inside the forward pass via the thermodynamic equation
3. Supervising on ΔH (direct label) and T_m (derived from ΔS via physics — no ΔS label!)
4. Adding a **soft physical penalty** that punishes violations of the second law (ΔH > 0, ΔS > 0, ΔG₃₇ > 0)

$$\mathcal{L} = \underbrace{\text{MSE}(\hat{\Delta H}, \Delta H)}_\text{enthalpy supervision} + \underbrace{\text{MSE}(\hat{T}_m, T_m)}_\text{Tm via physics} + \underbrace{\alpha \cdot \mathcal{P}}_\text{violation penalty}$$

### Inductive Biases Being Compared
| Model | Inductive Bias |
|-------|----------------|
| GNN (E0) | Permutation equivariance — local message passing |
| 2D CNN (E2) | Spatial translation invariance — folded ladder |
| SAT (E3) | Explicit relational attention — structural adjacency |
| **PINN (E4, this notebook)** | **Physical constraint — thermodynamic laws in loss** |

### Research Question
> Does forcing the model to learn the latent thermodynamic quantity ΔS (with no direct labels) and obey the Tm formula improve generalisation to out-of-distribution duplex sequences (the `ov` dataset)?

In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, math, time
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb

import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

COLORS = {
    'GNN':    '#7f8c8d',
    '1D_CNN': '#3498db',
    '2D_CNN': '#e74c3c',
    'SAT':    '#9b59b6',
    'PINN':   '#e67e22',   # orange for Physics-Informed
}
MODEL_NAME = 'PINN'

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
MAX_WIDTH = 15   # ceil(24/2) + 1 — matches 2Dconv.ipynb
NT_MAP    = {'A': 0, 'T': 1, 'G': 2, 'C': 3}

DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'

# Physical constants
# For intramolecular hairpin: Tm (K) = dH / dS  (concentration-independent)
# dH in kcal/mol, dS in kcal/(mol·K)
# Physically motivated dS bounds (training normalisation range)
DS_MIN = -0.25    # kcal/(mol·K)  — most stable hairpins
DS_MAX = -0.002   # kcal/(mol·K)  — barely stable

config = dict(
    model_name       = 'PINN_2DCNN',
    experiment_id    = 'E4',
    max_width        = MAX_WIDTH,
    in_channels      = 6,
    dropout          = 0.2,
    alpha_penalty    = 0.1,      # weight of physical violation penalty
    n_epoch          = 200,
    batch_size       = 256,
    lr               = 1e-3,
    weight_decay     = 1e-5,
    grad_clip        = 1.0,
    dataset          = 'arr',
    norm_method      = 'normalize',
    wandb_project    = 'NNN_Thesis_Experiments',
    checkpoint_dir   = 'MyExperiments/PINN/models',
)
print('Config loaded:', config)

Config loaded: {'model_name': 'PINN_2DCNN', 'experiment_id': 'E4', 'max_width': 15, 'in_channels': 6, 'dropout': 0.2, 'alpha_penalty': 0.1, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'norm_method': 'normalize', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/PINN/models'}


In [3]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)

with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH', 'Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH', 'Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH', 'Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

# Compute normalisation stats from TRAINING set only
sumstats = {
    'dH_min': float(train_df['dH'].min()), 'dH_max': float(train_df['dH'].max()),
    'Tm_min': float(train_df['Tm'].min()), 'Tm_max': float(train_df['Tm'].max()),
    'dS_min': DS_MIN,  'dS_max': DS_MAX,  # physics-derived bounds for ΔS
}

def normalize(val, mn, mx):   return (val - mn) / (mx - mn)
def unnormalize(val, mn, mx): return val * (mx - mn) + mn

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print(f'dH range: [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}] kcal/mol')
print(f'Tm range: [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}] °C')
print(f'dS range (physics): [{DS_MIN}, {DS_MAX}] kcal/(mol·K)')

Train: 25,025  |  Val: 1,318  |  Test: 1,387
dH range: [-68.2, -2.7] kcal/mol
Tm range: [13.6, 68.6] °C
dS range (physics): [-0.25, -0.002] kcal/(mol·K)


In [4]:
# ── 4. Encoding: 2D Folded Ladder (identical to 2Dconv.ipynb / E2) ─────────────
# This cell is intentionally a copy of the proven E2 encoding.

def encode_2d_hairpin(seq, struct, max_width=MAX_WIDTH):
    n_stem   = struct.count('(')
    n_loop   = struct.count('.')
    half_loop = n_loop // 2
    has_mid  = (n_loop % 2 == 1)
    fold_len = n_stem + half_loop

    top_seq    = seq[:fold_len]
    mid_nt     = seq[fold_len] if has_mid else None
    bot_seq    = seq[fold_len + (1 if has_mid else 0):][::-1]
    hbond      = [1.0] * n_stem + [0.0] * half_loop

    tensor = np.zeros((6, 3, max_width), dtype=np.float32)
    for i, nt in enumerate(top_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()], 0, i] = 1.0
    for i, nt in enumerate(bot_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()], 2, i] = 1.0
    for i, h in enumerate(hbond):
        tensor[4, 1, i] = h
    bb_col = fold_len
    if bb_col < max_width:
        tensor[5, 0, bb_col] = 1.0
        tensor[5, 1, bb_col] = 1.0
        tensor[5, 2, bb_col] = 1.0
    if has_mid and mid_nt and mid_nt.upper() in NT_MAP and bb_col < max_width:
        tensor[NT_MAP[mid_nt.upper()], 1, bb_col] = 1.0
    return tensor


def encode_2d_duplex(strand1, strand2, max_width=MAX_WIDTH):
    tensor = np.zeros((6, 3, max_width), dtype=np.float32)
    for i, nt in enumerate(strand1):
        if i < max_width and nt.upper() in NT_MAP:
            tensor[NT_MAP[nt.upper()], 0, i] = 1.0
    for i, nt in enumerate(strand2[::-1]):
        if i < max_width and nt.upper() in NT_MAP:
            tensor[NT_MAP[nt.upper()], 2, i] = 1.0
    for i in range(min(len(strand1), max_width)):
        tensor[4, 1, i] = 1.0
    return tensor


def encode_row_2d(row, max_width=MAX_WIDTH):
    refseq = str(row['RefSeq'])
    struct = str(row['TargetStruct'])
    if '+' in struct:
        plus_pos = struct.index('+')
        s1, s2 = refseq[:plus_pos], refseq[plus_pos+1:]
        arr = encode_2d_duplex(s1, s2, max_width)
    else:
        arr = encode_2d_hairpin(refseq, struct, max_width)
    return arr


# Sanity check
_s = df.iloc[0]
_x = encode_row_2d(_s)
print(f'Encoding shape: {_x.shape}  (6 channels × 3 rows × {MAX_WIDTH} width)')
print(f'Non-zero entries: {(_x > 0).sum()}')

Encoding shape: (6, 3, 15)  (6 channels × 3 rows × 15 width)
Non-zero entries: 25


In [5]:
# ── 5. Dataset & DataLoaders ──────────────────────────────────────────────────

class DNAPINNDataset(Dataset):
    """Returns (x_2d, y) where y = [dH_norm, Tm_norm] — same targets as E2."""

    def __init__(self, df, sumstats, max_width=MAX_WIDTH):
        self.df       = df.reset_index(drop=False)
        self.sumstats = sumstats
        self.max_width = max_width

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x   = encode_row_2d(row, self.max_width)   # (6, 3, W)

        dH_norm = normalize(row['dH'], self.sumstats['dH_min'], self.sumstats['dH_max'])
        Tm_norm = normalize(row['Tm'], self.sumstats['Tm_min'], self.sumstats['Tm_max'])
        y = np.array([dH_norm, Tm_norm], dtype=np.float32)

        return torch.tensor(x), torch.tensor(y)


train_ds = DNAPINNDataset(train_df, sumstats)
val_ds   = DNAPINNDataset(val_df,   sumstats)
test_ds  = DNAPINNDataset(test_df,  sumstats)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512,                  shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512,                  shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')
_x, _y = next(iter(train_loader))
print(f'Batch shapes — x: {_x.shape}  y: {_y.shape}')

Train batches: 98  |  Val batches: 3
Batch shapes — x: torch.Size([256, 6, 3, 15])  y: torch.Size([256, 2])


In [6]:
# ── 6. Model: PINN 2D CNN ─────────────────────────────────────────────────────
#
# SAME backbone as E2 (DNA_2DCNN). Only the output head changes:
#   E2 head → predicts [dH_norm, Tm_norm]  (two independent targets)
#   E4 head → predicts [dH_norm, dS_norm]  (enthalpy + entropy)
#
# The ThermodynamicsLayer converts [dH, dS] → Tm inside the forward pass.
# This means the gradient of the Tm loss flows BACK through the physics equation
# and shapes how the network learns dS — purely from the thermodynamic constraint.

class AttentionPool2d(nn.Module):
    """Learned attention pooling: (B,C,H,W) → (B,C)."""
    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=1),
            nn.Tanh(),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        scores  = self.attn(x).view(B, 1, -1)          # (B,1,H*W)
        weights = torch.softmax(scores, dim=-1)         # (B,1,H*W)
        return (x.view(B, C, -1) * weights).sum(-1)     # (B,C)


class ThermodynamicsLayer(nn.Module):
    """
    Non-learnable physics layer: converts normalised [dH_norm, dS_norm] → Tm_norm.

    Forward:
        1. Unnormalise dH, dS to physical units
        2. Tm (K)  = dH / dS           (intramolecular hairpin, concentration-independent)
        3. Tm (°C) = Tm_K - 273.15
        4. Normalise Tm to [0,1]

    Gradients flow through unnormalise → division → renormalise,
    so dS is shaped entirely by the Tm supervision signal.
    """

    def __init__(self, sumstats):
        super().__init__()
        # Register as buffers so they move with .to(device) automatically
        self.register_buffer('dH_min', torch.tensor(sumstats['dH_min'], dtype=torch.float32))
        self.register_buffer('dH_max', torch.tensor(sumstats['dH_max'], dtype=torch.float32))
        self.register_buffer('dS_min', torch.tensor(sumstats['dS_min'], dtype=torch.float32))
        self.register_buffer('dS_max', torch.tensor(sumstats['dS_max'], dtype=torch.float32))
        self.register_buffer('Tm_min', torch.tensor(sumstats['Tm_min'], dtype=torch.float32))
        self.register_buffer('Tm_max', torch.tensor(sumstats['Tm_max'], dtype=torch.float32))

    def forward(self, dH_norm, dS_norm):
        """
        dH_norm, dS_norm: (B,) tensors in normalised space.
        Returns Tm_norm: (B,) in [0,1] space (clipped to avoid extreme gradients).
        """
        dH = dH_norm * (self.dH_max - self.dH_min) + self.dH_min   # kcal/mol
        dS = dS_norm * (self.dS_max - self.dS_min) + self.dS_min   # kcal/(mol·K)

        # Clamp dS away from zero to prevent division explosion
        dS_safe = torch.where(dS < 0, dS.clamp(max=-1e-4), dS.clamp(min=1e-4))

        Tm_K  = dH / dS_safe          # Kelvin
        Tm_C  = Tm_K - 273.15         # Celsius
        Tm_norm = (Tm_C - self.Tm_min) / (self.Tm_max - self.Tm_min)
        return Tm_norm.clamp(-2.0, 3.0)   # soft clip — avoids extreme loss spikes early on


class PINN_2DCNN(nn.Module):
    """
    Physics-Informed 2D CNN.

    Backbone:   identical to E2 (DNA_2DCNN with AttentionPool2d)
    Head:       Linear(128→2) → [dH_norm, dS_norm]
    Physics:    ThermodynamicsLayer: [dH_norm, dS_norm] → Tm_norm
    Output:     [dH_norm, dS_norm, Tm_norm]  — 3 values
    """

    def __init__(self, sumstats, in_channels=6, dropout=0.2):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 64,  kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(64),  nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(64,        128, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(128,       128, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
        )
        self.attn_pool = AttentionPool2d(128)

        # Head predicts [dH_norm, dS_norm] — entropy head has no prior labels
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

        self.thermo = ThermodynamicsLayer(sumstats)

    def forward(self, x):
        """
        x: (B, 6, 3, W)
        Returns: dict with dH_norm, dS_norm, Tm_norm — all (B,)
        """
        feat   = self.attn_pool(self.backbone(x))   # (B, 128)
        out    = self.head(feat)                     # (B, 2)
        dH_n   = out[:, 0]
        dS_n   = out[:, 1]
        Tm_n   = self.thermo(dH_n, dS_n)
        return {'dH_norm': dH_n, 'dS_norm': dS_n, 'Tm_norm': Tm_n}


model = PINN_2DCNN(sumstats,
                   in_channels=config['in_channels'],
                   dropout=config['dropout']).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: PINN_2DCNN')
print(f'Total trainable parameters: {n_params:,}')
print(f'  Backbone: same as E2 (2D CNN with attention pooling)')
print(f'  Output head: 128 → 64 → [dH_norm, dS_norm]  (dS unsupervised — physics-only)')
print(f'  Physics layer: Tm = dH/dS - 273.15 → normalised')

Model: PINN_2DCNN
Total trainable parameters: 340,611
  Backbone: same as E2 (2D CNN with attention pooling)
  Output head: 128 → 64 → [dH_norm, dS_norm]  (dS unsupervised — physics-only)
  Physics layer: Tm = dH/dS - 273.15 → normalised


In [7]:
# ── 7. Metrics & Evaluation Helpers (standard across all notebooks) ───────────

def compute_metrics(pred_norm, true_norm, sumstats):
    """
    pred_norm, true_norm: (N, 2) arrays [dH_norm, Tm_norm]
    Returns metrics dict + unnormalized arrays.
    """
    if torch.is_tensor(pred_norm): pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm): true_norm = true_norm.cpu().numpy()

    dH_p = pred_norm[:, 0] * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min']
    Tm_p = pred_norm[:, 1] * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min']
    dH_t = true_norm[:, 0] * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min']
    Tm_t = true_norm[:, 1] * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min']

    dG_p = dH_p * (1.0 - (273.15 + 37.0) / (273.15 + Tm_p))
    dG_t = dH_t * (1.0 - (273.15 + 37.0) / (273.15 + Tm_t))

    metrics = {}
    for tag, p, t in [('dH', dH_p, dH_t), ('Tm', Tm_p, Tm_t), ('dG_37', dG_p, dG_t)]:
        mask = np.isfinite(t) & np.isfinite(p)
        if mask.sum() < 2:
            metrics[f'{tag}_mae'] = metrics[f'{tag}_rmse'] = metrics[f'{tag}_r2'] = float('nan')
        else:
            diff = p[mask] - t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(diff)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(diff ** 2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask], p[mask]))

    return metrics, dH_p, Tm_p, dH_t, Tm_t


@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    """Run PINN model over loader; convert physics output back to standard [dH, Tm] for metrics."""
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        # Stack predicted [dH_norm, Tm_norm] (Tm comes from physics)
        pred_pair = torch.stack([out['dH_norm'], out['Tm_norm']], dim=1).cpu()
        preds.append(pred_pair)
        trues.append(y)
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    metrics, dH_p, Tm_p, dH_t, Tm_t = compute_metrics(preds, trues, sumstats)
    return metrics, dH_p, Tm_p, dH_t, Tm_t


print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. PINN Loss Function ─────────────────────────────────────────────────────

class PINNLoss(nn.Module):
    """
    Physics-Informed Loss:

        L = MSE(dH_pred, dH_true)                   [enthalpy — direct label]
          + MSE(Tm_pred, Tm_true)                   [Tm derived from dH/dS physics]
          + alpha * P                               [soft physical violation penalty]

    Physical violation penalty P:
        p1: dH > 0     (hybridisation must be exothermic)
        p2: dS > 0     (hybridisation must decrease entropy)
        p3: dG_37 > 0  (native state must be stable at 37°C: dG = dH - 310.15*dS < 0)

    dH and dS are unnormalised inside the penalty for physically meaningful gradients.
    """

    def __init__(self, sumstats, alpha=0.1):
        super().__init__()
        self.alpha = alpha
        self.ss    = sumstats

    def forward(self, out, y_true):
        """
        out:    dict from PINN_2DCNN.forward()
        y_true: (B, 2) [dH_norm_true, Tm_norm_true]
        """
        dH_n = out['dH_norm']   # (B,)
        dS_n = out['dS_norm']   # (B,)
        Tm_n = out['Tm_norm']   # (B,)  — derived from physics

        # Supervised terms
        loss_dH = F.mse_loss(dH_n, y_true[:, 0])
        loss_Tm = F.mse_loss(Tm_n, y_true[:, 1])

        # Unnormalise for physical penalty (gradients in physical units)
        dH_phys = dH_n * (self.ss['dH_max'] - self.ss['dH_min']) + self.ss['dH_min']
        dS_phys = dS_n * (self.ss['dS_max'] - self.ss['dS_min']) + self.ss['dS_min']

        p1 = F.relu(dH_phys).mean()                     # dH must be negative
        p2 = F.relu(dS_phys).mean()                     # dS must be negative
        p3 = F.relu(dH_phys - 310.15 * dS_phys).mean() # dG_37 must be negative
        penalty = p1 + p2 + p3

        total = loss_dH + loss_Tm + self.alpha * penalty
        return total, loss_dH.item(), loss_Tm.item(), penalty.item()


criterion = PINNLoss(sumstats, alpha=config['alpha_penalty'])
print('PINN loss function instantiated.')
print(f'  alpha_penalty = {config["alpha_penalty"]}')

PINN loss function instantiated.
  alpha_penalty = 0.1


In [9]:
# ── 9. Training Loop ──────────────────────────────────────────────────────────

optimizer = optim.Adam(model.parameters(), lr=config['lr'],
                       weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['n_epoch'], eta_min=1e-5
)

history = {
    'train_loss': [], 'train_loss_dH': [], 'train_loss_Tm': [], 'train_penalty': [],
    'val_dH_mae': [], 'val_Tm_mae': [], 'val_dG_mae': [],
    'val_dH_rmse': [], 'val_Tm_rmse': [],
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

_run_name  = f"PINN_alpha{config['alpha_penalty']}"
_wandb_kw  = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_mode      = os.environ.get('WANDB_MODE', '').strip().lower()
if _mode in ('offline', 'disabled'):
    run = wandb.init(mode=_mode, **_wandb_kw)
else:
    try:
        run = wandb.init(**_wandb_kw)
    except Exception as _e:
        print(f'WandB online init failed ({_e}), falling back to offline.')
        run = wandb.init(mode='offline', **_wandb_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf')
start_time  = time.time()

for epoch in range(config['n_epoch']):
    model.train()
    ep_loss = ep_dH = ep_Tm = ep_pen = 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss, l_dH, l_Tm, pen = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        bs = x.size(0)
        ep_loss += loss.item() * bs
        ep_dH   += l_dH * bs
        ep_Tm   += l_Tm * bs
        ep_pen  += pen  * bs

    n_train = len(train_loader.dataset)
    ep_loss /= n_train; ep_dH /= n_train; ep_Tm /= n_train; ep_pen /= n_train
    scheduler.step()

    val_metrics, _, _, _, _ = evaluate(model, val_loader, sumstats, device)

    history['train_loss'].append(ep_loss)
    history['train_loss_dH'].append(ep_dH)
    history['train_loss_Tm'].append(ep_Tm)
    history['train_penalty'].append(ep_pen)
    history['val_dH_mae'].append(val_metrics['dH_mae'])
    history['val_Tm_mae'].append(val_metrics['Tm_mae'])
    history['val_dG_mae'].append(val_metrics['dG_37_mae'])
    history['val_dH_rmse'].append(val_metrics['dH_rmse'])
    history['val_Tm_rmse'].append(val_metrics['Tm_rmse'])

    wandb.log({
        'epoch':         epoch,
        'train_loss':    ep_loss,
        'train_loss_dH': ep_dH,
        'train_loss_Tm': ep_Tm,
        'train_penalty': ep_pen,
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'lr':            scheduler.get_last_lr()[0],
    })

    if val_metrics['dG_37_mae'] < best_val_dG:
        best_val_dG = val_metrics['dG_37_mae']
        torch.save(model.state_dict(),
                   os.path.join(config['checkpoint_dir'], 'best_pinn_model.pt'))

    if (epoch + 1) % 20 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} "
              f"| loss {ep_loss:.4f} (dH {ep_dH:.4f} Tm {ep_Tm:.4f} pen {ep_pen:.4f}) "
              f"| val dH {val_metrics['dH_mae']:.3f} Tm {val_metrics['Tm_mae']:.3f} dG {val_metrics['dG_37_mae']:.3f} "
              f"| {elapsed:.1f}min")

run.finish()

with open('out/pinn_history.json', 'w') as f:
    json.dump(history, f)

print(f'\n=== Training complete ===')
print(f'Best val dG_37 MAE: {best_val_dG:.4f} kcal/mol')
print(f'History saved to: out/pinn_history.json')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.


wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: PINN_alpha0.1  |  mode: online
Ep  20/200 | loss 0.0503 (dH 0.0197 Tm 0.0284 pen 0.0215) | val dH 5.693 Tm 5.275 dG 0.449 | 3.2min
Ep  40/200 | loss 0.0316 (dH 0.0148 Tm 0.0152 pen 0.0162) | val dH 4.562 Tm 4.887 dG 0.326 | 6.3min
Ep  60/200 | loss 0.0265 (dH 0.0133 Tm 0.0122 pen 0.0092) | val dH 5.089 Tm 3.482 dG 0.302 | 9.5min
Ep  80/200 | loss 0.0242 (dH 0.0114 Tm 0.0117 pen 0.0117) | val dH 4.310 Tm 3.987 dG 0.309 | 12.9min
Ep 100/200 | loss 0.0230 (dH 0.0107 Tm 0.0114 pen 0.0094) | val dH 4.383 Tm 3.472 dG 0.239 | 16.2min
Ep 120/200 | loss 0.0205 (dH 0.0099 Tm 0.0099 pen 0.0068) | val dH 3.916 Tm 3.625 dG 0.304 | 19.6min
Ep 140/200 | loss 0.0186 (dH 0.0092 Tm 0.0087 pen 0.0070) | val dH 3.615 Tm 2.880 dG 0.211 | 22.9min
Ep 160/200 | loss 0.0174 (dH 0.0088 Tm 0.0080 pen 0.0063) | val dH 3.536 Tm 2.911 dG 0.208 | 26.3min
Ep 180/200 | loss 0.0169 (dH 0.0085 Tm 0.0078 pen 0.0057) | val dH 3.573 Tm 2.858 dG 0.204 | 29.6min
Ep 200/200 | loss 0.0167 (dH 0.0084 Tm 0.0078 pen 0.

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇██████
lr,██████▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_Tm,▅█▅▅▄▄▄▄▄▄▃▃▃▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_dH,█▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_penalty,▄▄▄▆█▂▃▇▆▅▃▁▃▂▂▃▂▂▅▂▃▂▂▂▁▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,█▅▅▃▂▃▂▅▄▃▃▃▂▃▂▄▂▁▂▂▂▂▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_r2,▁▁▅▄▃▆▄▆▅▅▄▇▇▇▆▇▇▄▇▇▇▇▇▇▇▇▇▇▇███████████
val_Tm_rmse,▆█▅▆▄▃▄▂▃▂▃▂▂▂▂▃▂▁▂▂▂▂▁▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁
val_dG_37_mae,▄█▃▆▂▂▂▃▂▂▂▃▂▂▂▁▁▂▁▁▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...



=== Training complete ===
Best val dG_37 MAE: 0.1950 kcal/mol
History saved to: out/pinn_history.json


In [10]:
# ── 10. Final Evaluation ──────────────────────────────────────────────────────

model.load_state_dict(torch.load(
    os.path.join(config['checkpoint_dir'], 'best_pinn_model.pt'),
    map_location=device
))

val_metrics,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_metrics, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Validation Set Results (arr) ===')
for tag, key in [('dH', 'dH'), ('Tm', 'Tm'), ('dG₃₇', 'dG_37')]:
    print(f'  {tag}   MAE {val_metrics[f"{key}_mae"]:.3f}  RMSE {val_metrics[f"{key}_rmse"]:.3f}  R² {val_metrics[f"{key}_r2"]:.3f}')

print('\n=== Test Set Results (arr) ===')
for tag, key in [('dH', 'dH'), ('Tm', 'Tm'), ('dG₃₇', 'dG_37')]:
    print(f'  {tag}   MAE {test_metrics[f"{key}_mae"]:.3f}  RMSE {test_metrics[f"{key}_rmse"]:.3f}  R² {test_metrics[f"{key}_r2"]:.3f}')

# Save eval CSV
eval_df = pd.DataFrame({'dH_pred': dH_vp, 'dH_true': dH_vt,
                         'Tm_pred': Tm_vp, 'Tm_true': Tm_vt})
eval_df['dG_pred'] = eval_df['dH_pred'] * (1 - 310.15 / (273.15 + eval_df['Tm_pred']))
eval_df['dG_true'] = eval_df['dH_true'] * (1 - 310.15 / (273.15 + eval_df['Tm_true']))
eval_df.to_csv('out/pinn_val_eval.csv', index=False)

# Save run log for EXPERIMENT_LOG
run_log = dict(
    experiment_id='E4', model='PINN_2DCNN', config=config,
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
    val_metrics=val_metrics, test_metrics=test_metrics,
    best_checkpoint=os.path.join(config['checkpoint_dir'], 'best_pinn_model.pt'),
    history_path='out/pinn_history.json',
    eval_csv_path='out/pinn_val_eval.csv',
)
with open('out/pinn_run_log.json', 'w') as f:
    json.dump(run_log, f, indent=2)
print('\nRun log saved to: out/pinn_run_log.json')

=== Validation Set Results (arr) ===
  dH   MAE 3.597  RMSE 4.638  R² 0.823
  Tm   MAE 2.754  RMSE 3.950  R² 0.866
  dG₃₇   MAE 0.195  RMSE 0.266  R² 0.934

=== Test Set Results (arr) ===
  dH   MAE 3.612  RMSE 4.562  R² 0.828
  Tm   MAE 2.737  RMSE 3.899  R² 0.865
  dG₃₇   MAE 0.200  RMSE 0.268  R² 0.934

Run log saved to: out/pinn_run_log.json


In [11]:
# ── 11. Convergence Curves (F2 contribution) ──────────────────────────────────

os.makedirs('out/figures', exist_ok=True)
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='#f8f9fa')

# Row 1: standard val metrics
for ax, (key, ylabel) in zip(axes[0], [
    ('val_dH_mae',  'Val dH MAE (kcal/mol)'),
    ('val_Tm_mae',  'Val Tm MAE (°C)'),
    ('val_dG_mae',  'Val ΔG₃₇ MAE (kcal/mol)'),
]):
    ax.plot(epochs, history[key], color=COLORS['PINN'], lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    sns.despine(ax=ax)

# Row 2: PINN-specific loss components
for ax, (key, ylabel, color) in zip(axes[1], [
    ('train_loss_dH', 'Train Loss (ΔH term)', '#2ecc71'),
    ('train_loss_Tm', 'Train Loss (Tm term)', '#3498db'),
    ('train_penalty', 'Physical Violation Penalty', '#e74c3c'),
]):
    ax.plot(epochs, history[key], color=color, lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    sns.despine(ax=ax)

fig.suptitle('Figure F2 (partial) — PINN Convergence + Loss Decomposition',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/pinn_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/pinn_convergence.png')

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0, flags=flags)


Saved: out/figures/pinn_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_12616\3144586604.py:32: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [12]:
# ── 12. Scatter Plots — Predicted vs Measured (F3 contribution) ───────────────

AXIS_LIMITS = {'dH': (-55, -5), 'Tm': (20, 60), 'dG_37': (-7, 5)}

dG_vp = dH_vp * (1 - 310.15 / (273.15 + Tm_vp))
dG_vt = dH_vt * (1 - 310.15 / (273.15 + Tm_vt))

fig, axes = plt.subplots(1, 3, figsize=(14, 5), facecolor='#f8f9fa')
for ax, (pred_arr, true_arr, tag, unit) in zip(axes, [
    (dH_vp, dH_vt, 'dH',    'kcal/mol'),
    (Tm_vp, Tm_vt, 'Tm',    '°C'),
    (dG_vp, dG_vt, 'dG_37', 'kcal/mol'),
]):
    lim = AXIS_LIMITS[tag]
    ax.scatter(true_arr, pred_arr, s=4, alpha=0.4, color=COLORS['PINN'], rasterized=True)
    ax.plot(lim, lim, 'k--', alpha=0.3, lw=1.5)
    mask = np.isfinite(pred_arr) & np.isfinite(true_arr)
    mae  = np.mean(np.abs(pred_arr[mask] - true_arr[mask]))
    rmse = np.sqrt(np.mean((pred_arr[mask] - true_arr[mask]) ** 2))
    r2   = r2_score(true_arr[mask], pred_arr[mask])
    ax.text(0.05, 0.93, f'MAE={mae:.3f}\nRMSE={rmse:.3f}\nR²={r2:.3f}',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f'Measured {tag} ({unit})'); ax.set_ylabel(f'Predicted {tag} ({unit})')
    ax.set_title(f'PINN — {tag}', fontsize=11, fontweight='bold')
    sns.despine(ax=ax)

fig.suptitle('Figure F3 (partial) — PINN: Predicted vs Measured (Validation)', fontsize=12)
plt.tight_layout()
plt.savefig('out/figures/pinn_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/pinn_scatter.png')

Saved: out/figures/pinn_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_12616\1919110431.py:32: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
# ── 13. Physics Analysis (F6) — Does the PINN respect thermodynamic laws? ─────
#
# Key thesis figure: show that the learned ΔS predictions are physically
# meaningful even though ΔS was NEVER directly supervised.
#
# Classic cross-check: plot predicted Tm (from dH/dS physics) vs. true Tm,
# and plot predicted ΔS vs. the pseudo-ΔS derived from labels:
#    dS_pseudo = dH_true / (Tm_true + 273.15)

@torch.no_grad()
def get_pinn_latents(model, loader, sumstats, device):
    """Collect dH_pred, dS_pred, Tm_pred (all physical units) for a loader."""
    model.eval()
    dH_p, dS_p, Tm_p, dH_t, Tm_t = [], [], [], [], []
    for x, y in loader:
        out = model(x.to(device))
        dH_n = out['dH_norm'].cpu().numpy()
        dS_n = out['dS_norm'].cpu().numpy()
        Tm_n = out['Tm_norm'].cpu().numpy()

        dH_p.append(dH_n * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min'])
        dS_p.append(dS_n * (sumstats['dS_max'] - sumstats['dS_min']) + sumstats['dS_min'])
        Tm_p.append(Tm_n * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min'])

        dH_t.append(y[:, 0].numpy() * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min'])
        Tm_t.append(y[:, 1].numpy() * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min'])

    return (np.concatenate(dH_p), np.concatenate(dS_p), np.concatenate(Tm_p),
            np.concatenate(dH_t), np.concatenate(Tm_t))


dH_p, dS_p, Tm_p, dH_t, Tm_t = get_pinn_latents(model, val_loader, sumstats, device)

# Pseudo-ΔS from labels (the 'true' ΔS under the simplified hairpin formula)
dS_pseudo = dH_t / (Tm_t + 273.15)   # kcal/(mol·K)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='#f8f9fa')

# Panel 1: predicted vs pseudo ΔS  (key PINN thesis plot)
mask = np.isfinite(dS_pseudo) & np.isfinite(dS_p)
axes[0].scatter(dS_pseudo[mask], dS_p[mask], s=4, alpha=0.4, color=COLORS['PINN'], rasterized=True)
lim_s = (dS_pseudo[mask].min() * 1.1, dS_pseudo[mask].max() * 0.9)
axes[0].plot([lim_s[0], lim_s[1]], [lim_s[0], lim_s[1]], 'k--', alpha=0.3, lw=1.5)
r2_s = r2_score(dS_pseudo[mask], dS_p[mask])
axes[0].text(0.05, 0.93, f'R²={r2_s:.3f}', transform=axes[0].transAxes,
             fontsize=9, va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
axes[0].set_xlabel('Pseudo-ΔS from labels (kcal/mol·K)')
axes[0].set_ylabel('Predicted ΔS (kcal/mol·K)')
axes[0].set_title('Latent ΔS: no direct supervision\n(learned purely from thermodynamic law)',
                  fontsize=10, fontweight='bold')
sns.despine(ax=axes[0])

# Panel 2: histogram of predicted ΔS — should be entirely negative
pct_neg = (dS_p < 0).mean() * 100
axes[1].hist(dS_p, bins=60, color=COLORS['PINN'], edgecolor='white', lw=0.3, alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='ΔS = 0 (physical boundary)')
axes[1].text(0.05, 0.93, f'{pct_neg:.1f}% predictions ΔS < 0',
             transform=axes[1].transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
axes[1].set_xlabel('Predicted ΔS (kcal/mol·K)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Predicted ΔS\n(must be < 0 for physical validity)',
                  fontsize=10, fontweight='bold')
axes[1].legend(fontsize=8)
sns.despine(ax=axes[1])

# Panel 3: predicted dG_37 distribution — should be negative for all stable hairpins
dG_p_phys = dH_p * (1 - 310.15 / (273.15 + Tm_p))
pct_neg_dg = (dG_p_phys < 0).mean() * 100
axes[2].hist(dG_p_phys, bins=60, color='#2ecc71', edgecolor='white', lw=0.3, alpha=0.85)
axes[2].axvline(0, color='red', lw=1.5, linestyle='--', label='ΔG = 0')
axes[2].text(0.05, 0.93, f'{pct_neg_dg:.1f}% predictions ΔG₃₇ < 0',
             transform=axes[2].transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
axes[2].set_xlabel('Predicted ΔG₃₇ (kcal/mol)')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Predicted ΔG₃₇\n(must be < 0 for stable structure)',
                  fontsize=10, fontweight='bold')
axes[2].legend(fontsize=8)
sns.despine(ax=axes[2])

fig.suptitle('Figure F6 — PINN Physical Constraint Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/pinn_physics_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/pinn_physics_analysis.png')
print(f'\nPhysical validity summary:')
print(f'  ΔS < 0: {pct_neg:.1f}% of predictions')
print(f'  ΔG₃₇ < 0: {pct_neg_dg:.1f}% of predictions')
print(f'  Latent ΔS R² vs pseudo-labels: {r2_s:.3f}')

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0, flags=flags)


Saved: out/figures/pinn_physics_analysis.png

Physical validity summary:
  ΔS < 0: 100.0% of predictions
  ΔG₃₇ < 0: 84.6% of predictions
  Latent ΔS R² vs pseudo-labels: 0.808


C:\Users\anant\AppData\Local\Temp\ipykernel_12616\4135738279.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
